# Matplotlib 2/3 : Mise en forme et plusieurs graphiques

**Objectifs** : disposer plusieurs graphiques dans une même figure (`subplots`), personnaliser les axes, ajouter des annotations, tracer des diagrammes en barres et utiliser des couleurs pour représenter une troisième variable.

*Durée estimée : 40 min.*

In [ ]:
# Cellule de préparation : à exécuter en premier (ne fait rien en local)
import os, sys
if "google.colab" in sys.modules and not os.path.exists("intro-data-python"):
    !git clone -q https://github.com/VOTRE-COMPTE/intro-data-python.git
    %cd intro-data-python/notebooks

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/titanic.csv")

## 1. Plusieurs graphiques : `plt.subplots(lignes, colonnes)`

`plt.subplots(1, 2)` crée une figure avec 1 ligne et 2 colonnes d'axes. On récupère un tableau `axes` qu'on indexe.

In [ ]:
x = np.linspace(0, 2 * np.pi, 100)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(x, np.sin(x), color="tab:blue")
axes[0].set_title("sin")

axes[1].plot(x, np.cos(x), color="tab:red")
axes[1].set_title("cos")

plt.show()

- `figsize=(largeur, hauteur)` est en pouces.
- Avec une grille 2D (`plt.subplots(2, 2)`), `axes[i, j]` désigne la case ligne i, colonne j.
- `constrained_layout=True` évite que les titres et étiquettes se chevauchent.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 7), constrained_layout=True)

axes[0, 0].plot(x, np.sin(x));   axes[0, 0].set_title("sin")
axes[0, 1].plot(x, np.cos(x));   axes[0, 1].set_title("cos")
axes[1, 0].plot(x, x**2);        axes[1, 0].set_title("x²")
axes[1, 1].plot(x, np.exp(x));   axes[1, 1].set_title("exp")

fig.suptitle("Quatre fonctions", fontsize=14)
plt.show()

`sharex=True` / `sharey=True` partagent les axes entre graphiques (utile pour comparer à la même échelle).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].hist(df[df["Sex"] == "male"]["Age"].dropna(), bins=15)
axes[0].set_title("Hommes")
axes[1].hist(df[df["Sex"] == "female"]["Age"].dropna(), bins=15)
axes[1].set_title("Femmes")
plt.show()

## 2. Personnaliser les axes

In [ ]:
x = np.linspace(-5, 5, 200)

fig, ax = plt.subplots()
ax.plot(x, np.tanh(x))
ax.set_xlim(-4, 4)          # bornes de l'axe x
ax.set_ylim(-1.5, 1.5)      # bornes de l'axe y
ax.set_xticks([-4, -2, 0, 2, 4])   # graduations choisies
ax.grid(True, linestyle=":")
plt.show()

Échelle logarithmique : `ax.set_yscale("log")` (ou `set_xscale`). Très utile quand les valeurs s'étalent sur plusieurs ordres de grandeur.

In [ ]:
x = np.linspace(0, 10, 100)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(x, np.exp(x));  axes[0].set_title("Échelle linéaire")
axes[1].plot(x, np.exp(x));  axes[1].set_yscale("log");  axes[1].set_title("Échelle logarithmique")
plt.show()

## 3. Annotations et repères

| Méthode | Rôle |
|---|---|
| `ax.axhline(y)` / `ax.axvline(x)` | ligne horizontale / verticale |
| `ax.text(x, y, "texte")` | texte à une position |
| `ax.annotate("texte", xy=..., xytext=..., arrowprops=...)` | texte avec flèche |
| `ax.fill_between(x, y1, y2)` | zone colorée entre deux courbes |

In [ ]:
x = np.linspace(0, 2 * np.pi, 200)
y = np.sin(x)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y)
ax.axhline(0, color="gray", linewidth=0.8)
ax.fill_between(x, y, 0, where=(y > 0), alpha=0.3, color="tab:green")
ax.annotate("maximum", xy=(np.pi / 2, 1), xytext=(2.5, 0.9), arrowprops=dict(arrowstyle="->"))
ax.set_title("Sinus avec annotations")
plt.show()

## 4. Diagrammes en barres : `bar` et `barh`

Pour comparer des quantités entre catégories. On prépare souvent les données avec un `groupby` !

In [ ]:
taux = df.groupby("Pclass")["Survived"].mean()
print(taux)

fig, ax = plt.subplots()
barres = ax.bar(taux.index.astype(str), taux.values, color=["gold", "silver", "peru"])
ax.bar_label(barres, fmt="%.2f")    # valeurs au-dessus des barres
ax.set_xlabel("Classe")
ax.set_ylabel("Taux de survie")
ax.set_ylim(0, 1)
ax.set_title("Taux de survie selon la classe")
plt.show()

In [ ]:
# Barres horizontales : pratique quand les étiquettes sont longues
ports = df["Embarked"].value_counts()
fig, ax = plt.subplots()
ax.barh(ports.index, ports.values)
ax.set_xlabel("Nombre de passagers")
ax.set_ylabel("Port")
plt.show()

Barres **groupées** : on décale les positions manuellement.

In [ ]:
t = df.groupby(["Pclass", "Sex"])["Survived"].mean().unstack()
largeur = 0.35
pos = np.arange(len(t))

fig, ax = plt.subplots()
ax.bar(pos - largeur / 2, t["female"], largeur, label="Femmes")
ax.bar(pos + largeur / 2, t["male"], largeur, label="Hommes")
ax.set_xticks(pos)
ax.set_xticklabels(["1ère", "2ème", "3ème"])
ax.set_ylabel("Taux de survie")
ax.legend()
plt.show()

## 5. La couleur comme troisième variable

Avec `scatter(x, y, c=valeurs, cmap=...)`, la couleur de chaque point encode une valeur numérique. On ajoute une **barre de couleur** avec `fig.colorbar`.

In [ ]:
d = df.dropna(subset=["Age"])

fig, ax = plt.subplots()
sc = ax.scatter(d["Age"], d["Fare"], c=d["Pclass"], cmap="viridis", alpha=0.7)
fig.colorbar(sc, ax=ax, label="Classe", ticks=[1, 2, 3])
ax.set_xlabel("Âge")
ax.set_ylabel("Tarif")
plt.show()

Quelques cartes de couleurs : `viridis` (séquentielle, défaut recommandé), `plasma`, `coolwarm` (divergente), `tab10` (catégorielle). La liste complète est dans la documentation de Matplotlib (*colormap reference*).

## 6. Styles prédéfinis

Un style change l'apparence globale en une ligne. Voir `plt.style.available` pour la liste.

In [ ]:
with plt.style.context("ggplot"):
    fig, ax = plt.subplots()
    ax.plot(np.linspace(0, 1, 50), np.linspace(0, 1, 50) ** 2)
    ax.set_title("Style ggplot")
    plt.show()

## Récapitulatif

| Je veux… | Code |
|---|---|
| grille de graphiques | `fig, axes = plt.subplots(2, 2, figsize=(...), constrained_layout=True)` |
| titre général | `fig.suptitle("...")` |
| bornes, graduations | `ax.set_xlim`, `ax.set_ylim`, `ax.set_xticks` |
| échelle log | `ax.set_yscale("log")` |
| repères, annotations | `ax.axhline`, `ax.annotate`, `ax.fill_between` |
| barres | `ax.bar`, `ax.barh`, `ax.bar_label` |
| couleur = valeur | `scatter(..., c=..., cmap=...)` + `fig.colorbar` |

---
## Exercices

### ✏️ Exercice 1

Créez une figure 1×3 avec `sin(x)`, `cos(x)` et `sin(x) * cos(x)` pour `x` entre 0 et 2π. Donnez un titre à chaque graphique et un titre général.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
x = np.linspace(0, 2 * np.pi, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
axes[0].plot(x, np.sin(x)); axes[0].set_title("sin")
axes[1].plot(x, np.cos(x)); axes[1].set_title("cos")
axes[2].plot(x, np.sin(x) * np.cos(x)); axes[2].set_title("sin·cos")
fig.suptitle("Trois fonctions")
plt.show()
```

</details>

### ✏️ Exercice 2

Tracez un diagramme en barres du **tarif moyen** par port d'embarquement (`Embarked`), avec les valeurs affichées au-dessus des barres.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
m = df.groupby("Embarked")["Fare"].mean()
fig, ax = plt.subplots()
b = ax.bar(m.index, m.values)
ax.bar_label(b, fmt="%.1f")
ax.set_xlabel("Port")
ax.set_ylabel("Tarif moyen")
plt.show()
```

</details>

### ✏️ Exercice 3

Avec le fichier de températures, tracez Paris et Lyon sur un même graphique, colorez en gris clair la zone entre les deux courbes (`fill_between`) et ajoutez une ligne horizontale à 10 °C.

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
temp = pd.read_csv("../data/temperatures.csv")
fig, ax = plt.subplots()
ax.plot(temp["Mois"], temp["Paris"], label="Paris")
ax.plot(temp["Mois"], temp["Lyon"], label="Lyon")
ax.fill_between(temp["Mois"], temp["Paris"], temp["Lyon"], color="lightgray")
ax.axhline(10, color="k", linestyle="--", linewidth=1)
ax.legend()
plt.show()
```

</details>

### ✏️ Exercice 4

Nuage de points `Age` (x) vs `Fare` (y) pour les passagers de 1ère classe seulement, avec la couleur indiquant `Survived` (utilisez `cmap="coolwarm"` et une colorbar).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
p = df[(df["Pclass"] == 1)].dropna(subset=["Age"])
fig, ax = plt.subplots()
sc = ax.scatter(p["Age"], p["Fare"], c=p["Survived"], cmap="coolwarm", alpha=0.8)
fig.colorbar(sc, ax=ax, label="Survived")
ax.set_xlabel("Âge")
ax.set_ylabel("Tarif")
plt.show()
```

</details>

### ✏️ Exercice 5

**Défi** : tracez `sin(x)` sur [0, 2π] et ajoutez une annotation avec une flèche pointant sur son minimum (calculé avec `argmin`, pas à la main).

In [ ]:
# Votre code ici


<details>
<summary>💡 Voir la solution</summary>

```python
x = np.linspace(0, 2 * np.pi, 200)
y = np.sin(x)
i = y.argmin()
fig, ax = plt.subplots()
ax.plot(x, y)
ax.annotate("minimum", xy=(x[i], y[i]), xytext=(x[i] - 2, y[i] + 0.5), arrowprops=dict(arrowstyle="->"))
plt.show()
```

</details>